# Section 1: Spark and Iceberg on Cloudera

![alt text](https://raw.githubusercontent.com/richard-vh/cloudera-iceberg-spark-hol/refs/heads/main/content/assets/images/spark-iceberg-cldr.png)

## Getting Started

### Please set your username before initializing your Spark session. This simple step prefixes your tables and ensures your work doesn't conflict with others in the shared environment.


In [ ]:
# Enter your assigned user below
username = "user003"

from datetime import datetime
import cml.data_v1 as cmldata

SPARK_CONNECTION_NAME = "hol-aw-dl"
conn_spark = cmldata.get_connection(SPARK_CONNECTION_NAME)
spark = conn_spark.get_spark_session()

print("Code block completed")

## Section 1. Creating Iceberg Tables 

### Creating an Iceberg Table

**What is an Iceberg Table?**
An Iceberg table is an open-source table format designed for huge analytic datasets in data lakes. Think of it not as a new data storage format, but as a sophisticated management layer that sits on top of your existing data files (like Parquet, Avro, or ORC) in cloud storage like Amazon S3 or Google Cloud Storage. It brings the reliability and simplicity of traditional SQL tables to the world of big data. At its core, an Iceberg table tracks a list of data files that constitute the table's contents. Unlike older formats like Hive, which rely on tracking files at a directory level, Iceberg tracks individual files. This seemingly small difference is what unlocks a host of powerful features.

**Key benefits**

* **ACID Transactions**: Iceberg brings atomicity, consistency, isolation, and durability to your data lake. This means that operations are either fully completed or not at all, ensuring that your data remains in a consistent state, even with multiple users reading and writing to the same table simultaneously.
* **Schema Evolution**: With Iceberg, you can safely add, drop, rename, and even reorder columns in your table without needing to rewrite all the underlying data. This is a massive improvement over older formats where schema changes were often risky and computationally expensive.
* **Hidden Partitioning**: You can change the way your table is partitioned without breaking existing queries. Iceberg handles the partitioning logic behind the scenes, so users don't need to know the physical layout of the data to write efficient queries.
* **Time Travel and Rollback**: Every change to an Iceberg table creates a new, immutable snapshot of the table's state. This allows you to query the table as it existed at a specific point in time, which is invaluable for audits, debugging, and reproducing analyses. It also makes it easy to roll back the table to a previous version if a mistake is made.
* **Efficient Query Performance**: By storing detailed statistics about the data at the file level, Iceberg enables query engines to perform "file pruning." This means the engine can quickly identify and skip reading files that don't contain data relevant to a query, leading to significant performance gains.


In [ ]:
spark.sql("DROP TABLE IF EXISTS default.{}_managed_countries".format(username))

# Create an Iceberg table for European countries (MOR by default)
spark.sql("""
    CREATE TABLE default.{}_managed_countries (
        country_code STRING,
        country_name STRING,
        population INT,
        area DOUBLE
    )
    USING iceberg
""".format(username))

# Insert data into the table
spark.sql("""
    INSERT INTO default.{}_managed_countries VALUES
    ('FR', 'France', 67391582, 643801.0),
    ('DE', 'Germany', 83149300, 357022.0),
    ('IT', 'Italy', 60262770, 301340.0)
""".format(username))

print("Code block completed")

In [ ]:
# Query the table
df = spark.sql("SELECT * FROM default.{}_managed_countries".format(username))
df.show()

print("Code block completed")

3. Add a new cell to the notebook and run each code block to look at the properties of the table.

In [ ]:
# Describe the table
spark.sql("DESCRIBE default.{}_managed_countries".format(username)).show()

# Show the table's CREATE statement
spark.sql("SHOW CREATE TABLE default.{}_managed_countries".format(username)).show(truncate=False)

# Show the table properties
spark.sql("SHOW TBLPROPERTIES default.{}_managed_countries".format(username)).show(truncate=False)

print("Code block completed")

Describing the tables shows us the column names, their data types and column comments.

The create table statement give us the the current table DDL, including the storage location (note the S3 storage path, Iceberg current snapshot id, storage file type, format-version which indicates if it's an Iceberg v1 or v2 table.

## Section 2. Iceberg data Manipulation 

### Best Practices for Managing Data
  * Start with a comprehensive data model
  * Ensure that the schema is well-defined and follows consistent naming conventions.
  * Leverage nested structures for complex data

### Iceberg Data Inserts and Updates

In Iceberg, you can insert and update data using SQL commands. Inserts add new records to the table, while updates modify existing records based on a condition.

1. In your existing Jupyter notebook add a new cell and run the code below.

In [ ]:
# Drop the table if it exists
spark.sql("DROP TABLE IF EXISTS default.{}_english_football_teams".format(username))

# Create the table for football teams in England
spark.sql("""
    CREATE TABLE default.{}_english_football_teams (
        team_id STRING,
        team_name STRING,
        team_city STRING,
        team_stadium STRING
    )
    USING iceberg
""".format(username))

# Insert data into the table
spark.sql("""
    INSERT INTO default.{}_english_football_teams 
    VALUES 
    ('T001', 'Manchester United', 'Manchester', 'Old Trafford'),
    ('T002', 'Liverpool', 'Liverpool', 'Anfield'),
    ('T003', 'Chelsea', 'London', 'Stamford Bridge')
""".format(username))

df = spark.sql("SELECT * FROM default.{}_english_football_teams".format(username))
df.show(truncate=False)

# Update data for a football team
spark.sql("""
    UPDATE default.{}_english_football_teams
    SET team_stadium = 'New Stamford Bridge'
    WHERE team_id = 'T003'
""".format(username))

df = spark.sql("SELECT * FROM default.{}_english_football_teams".format(username))
df.show(truncate=False)

print("Code block completed")

2. Look at the code and output and verify the inserts and update work as you would expect.

### Iceberg Data Deletion

When performing deletions in Iceberg, it’s important to remember that Iceberg uses a snapshot mechanism. Deletions will add a new snapshot but do not immediately remove data, ensuring that deleted data can still be recovered.

#### What to Consider When Performing Deletions
  * Deletions in Iceberg are versioned and can be reverted through time travel.
  * Data is deleted based on conditions you specify (e.g., by team_id).
  * You can perform data compaction after deletion for performance optimization.

1. In your existing Jupyter notebook add a new cell and run the code below.

In [ ]:
# Deleting data from the table (removing Chelsea)
spark.sql("""
    DELETE FROM default.{}_english_football_teams
    WHERE team_id = 'T003'
""".format(username))

df = spark.sql("SELECT * FROM default.{}_english_football_teams".format(username))
df.show(truncate=False)

print("Code block completed")

2.  Look at the code and output and verify the delete works as you would expect.

## Section 3: Schema and Partition Evolution

### Iceberg Schema Evolution
Schema evolution in Iceberg allows you to modify the structure of your tables over time. This includes adding, renaming, and removing columns while ensuring that historical data remains accessible without requiring a full rewrite of the table.

**Iceberg supports schema evolution operations like:**
  * Adding columns: You can add new columns without affecting the existing data or operations.
  * Renaming columns: The renaming of columns is supported without requiring data migration.
  * Changing column types: You can change the type of a column, as long as it is compatible with the existing data.
  * Dropping columns: Columns can be safely dropped if they are no longer needed.

**Why is Schema Evolution important?**
  * Adapting to business needs: As your data requirements evolve, schema changes are often necessary without disrupting production workflows.
  * Backwards compatibility: Allows for schema changes that are compatible with existing data, meaning that you can evolve the schema without breaking old queries or affecting historical data.
  * Simplifying data management: Allows incremental changes to the schema without needing full table rewrites.


**Code Example:**

In your existing Jupyter notebook add a new cell and run the code below. Examine each statement and it's output to understand how the table's schema definition is changed by adding a new column and renaming an existing one without affecting existing data or having to migrate the data.

In [ ]:
spark.sql("""
	DROP TABLE IF EXISTS default.{}_zoo_animals_schema_evo
""".format(username))

# Create the initial Iceberg table
spark.sql("""
    CREATE TABLE default.{}_zoo_animals_schema_evo (
        animal_id STRING,
        animal_name STRING
    )
    USING iceberg
""".format(username))

# Insert sample data
spark.sql("""
    INSERT INTO default.{}_zoo_animals_schema_evo VALUES 
    ('A001', 'Lion'),
    ('A002', 'Elephant'),
    ('A003', 'Giraffe')
""".format(username))

print("---------------------------")
print("--BEFORE SCHEMA EVOLUTION--")
print("---------------------------")

# View the Data
df = spark.sql("select * from default.{}_zoo_animals_schema_evo".format(username))
df.show(truncate=False)

# View the schema
df = spark.sql("DESCRIBE TABLE default.{}_zoo_animals_schema_evo".format(username))
df.show(truncate=False)

# Add a new column to the table
spark.sql("""
    ALTER TABLE default.{}_zoo_animals_schema_evo ADD COLUMN habitat STRING
""".format(username))

# Rename an existing column
spark.sql("""
    ALTER TABLE default.{}_zoo_animals_schema_evo RENAME COLUMN animal_name TO species_name
""".format(username))

# Insert new data into the updated schema
spark.sql("""
    INSERT INTO default.{}_zoo_animals_schema_evo VALUES 
    ('A004', 'Zebra', 'Savanna'),
    ('A005', 'Panda', 'Bamboo Forest')
""".format(username))

print("---------------------------")
print("--AFTER SCHEMA EVOLUTION--")
print("---------------------------")

# View the Data
df = spark.sql("select * from default.{}_zoo_animals_schema_evo".format(username))
df.show(truncate=False)

# View the schema
df = spark.sql("DESCRIBE TABLE default.{}_zoo_animals_schema_evo".format(username))
df.show(truncate=False)

print("Code block completed")

### Iceberg Partition Evolution

**What is Partition Evolution?**
Partition evolution refers to the ability to modify the partitioning strategy of an Iceberg table after it has been created. This can involve changing the partitioning key (column used for partitioning) or adding new partitioning columns. Unlike traditional partitioning schemes, Iceberg allows for flexible partition evolution without needing to rewrite the entire dataset. Partitioning evolution can help optimize query performance and manage large datasets more efficiently.

**Common Partitioning Strategies**
  * Time-based partitioning: Commonly used for time-series data, partitioning by date or timestamp can help in partition pruning, making queries faster.
  * Range or hash partitioning: For datasets with discrete values like animal species, partitioning by a range of values or using hash partitioning can help balance the data across partitions.

You can change the partitioning strategy after the table has been created, even if the data already exists. This allows you to optimize partitioning as your query patterns evolve and take advantage of query optimization with regards to partition pruning as the data evolves and grows within the table.

**Code Example:**

In your existing Jupyter notebook add a new cell and run the code below. Examine each statement and it's output to understand how the table's patition strategy evolves.

In [ ]:
spark.sql("""
	DROP TABLE IF EXISTS default.{}_zoo_animals_partition_evo
""".format(username))

# Create the initial Iceberg table partitioned by 'animal_id'
spark.sql("""
    CREATE TABLE default.{}_zoo_animals_partition_evo (
        animal_id STRING,
        species_name STRING,
        habitat STRING
    )
    USING iceberg
    PARTITIONED BY (animal_id)
""".format(username))

# Insert data into the initial partitioning scheme
spark.sql("""
    INSERT INTO default.{}_zoo_animals_partition_evo VALUES
    ('A001', 'Lion', 'Savanna'),
    ('A002', 'Tiger', 'Forest')
""".format(username))

print("-----------------------------")
print("--BEFORE PARTITION EVOLUTION--")
print("-----------------------------")

# Querying data
spark.sql("""
    SELECT * FROM default.{}_zoo_animals_partition_evo
""".format(username)).show()

# Show the formatted schema and partitioning scheme before change
spark.sql("DESCRIBE FORMATTED default.{}_zoo_animals_partition_evo".format(username)).show(truncate=False)

# Change the partitioning scheme to partition by both 'animal_id' and 'habitat'
spark.sql("""ALTER TABLE default.{}_zoo_animals_partition_evo ADD PARTITION FIELD habitat""".format(username))

# Insert data after the partitioning scheme change
spark.sql("""
    INSERT INTO default.{}_zoo_animals_partition_evo VALUES
    ('A003', 'Elephant', 'Grassland'),
    ('A004', 'Panda', 'Mountain')
""".format(username))

print("-----------------------------")
print("--AFTER PARTITION EVOLUTION--")
print("-----------------------------")

# Querying data
spark.sql("""
    SELECT * FROM default.{}_zoo_animals_partition_evo
""".format(username)).show()

# Show the formatted schema and partitioning scheme after the change
spark.sql("DESCRIBE FORMATTED default.{}_zoo_animals_partition_evo".format(username)).show(truncate=False)

print("Code block completed")

## Section 4: Iceberg Time Travel & Rollbacks using Snapshots

### Understanding Time Travel in Iceberg

**What is Time Travel in Iceberg?**
Time travel in Iceberg allows you to query a table as it existed at a specific point in time in the past. This feature leverages Iceberg's snapshot-based architecture to track all changes made to the data over time. When you perform time travel, Iceberg will provide data based on the state of the table at a specified snapshot or timestamp.
Time travel is supported by specifying a timestamp or snapshotid when querying the table, which enables access to historical data without having to maintain separate copies of the data.

**How Time Travel enhances data management**
Historical queries: Time travel enables you to perform analysis on data as it existed at any previous time. This is valuable for auditing, debugging, and investigating historical trends.
Data recovery: In case of accidental data loss or corruption, you can use time travel to recover previous versions of data.
Simplifying rollbacks: Instead of reloading or restoring data from backups, you can simply query an earlier snapshot, reducing the complexity of data restoration.

**Key use cases and limitations**
  * Key use cases:
	* Auditing: Access historical data to verify what changes were made over time.
	* Data consistency: Ensure that queries are consistent with data at specific points in time, even in dynamic environments.
	* Bug investigation: Time travel is useful when tracking data issues or discrepancies in reports over time.
  * Limitations:
	* Snapshot retention: Older snapshots can be expired to optimize storage, which may impact time travel for long periods in the past.
	* Performance: Querying historical data may take longer depending on the size and number of snapshots involved.
	* Storage costs: Maintaining historical snapshots may increase storage costs.

**Code Example:**

In your existing Jupyter notebook add a new cell and run the code below. Examine each statement and it's output to understand how DML operations create snapshots and how we can use these snapshots to do time travel.

In [ ]:
spark.sql("""
	DROP TABLE IF EXISTS default.{}_european_cars_time_travel
""".format(username))

# Create an Iceberg table for European cars
spark.sql("""
    CREATE TABLE default.{}_european_cars_time_travel (
        car_id STRING,
        car_make STRING,
        car_model STRING,
        car_country STRING
    )
    USING iceberg
""".format(username))

# Insert initial data
spark.sql("""
    INSERT INTO default.{}_european_cars_time_travel VALUES 
    ('C001', 'Volkswagen', 'Golf', 'Germany'),
    ('C002', 'BMW', 'X5', 'Germany')
""".format(username))

# Insert new car data (Look, it's not a European Car)
spark.sql("""
    INSERT INTO default.{}_european_cars_time_travel VALUES 
    ('C003', 'FORD', 'F150', 'USA')
""".format(username))

print("-----------------------------")
print("--NOW WE HAVE 2 SNAPSHOTS WHICH REPRESENT THE TWO WRITE OPERATIONS (Once for German Cars and again for US Cars--")
print("-----------------------------")

# We now have incorrect data in the table
df = spark.sql("SELECT * FROM default.{}_european_cars_time_travel".format(username))
df.show(truncate=False)

snapshots_df = spark.sql("""
    SELECT * FROM default.{}_european_cars_time_travel.snapshots
""".format(username))
snapshots_df.show()

print("-----------------------------")
print("--WE CAN THEN TIMETRAVEL BACK TO THE TIME WERE FORD DID NOT EXIST--")
print("-----------------------------")

# Perform Time Travel to see the table before the edit
# Fetch the snapshot ID from the snapshot listing
rollback_snapshot_id_0 = snapshots_df.collect()[0].snapshot_id  
print(rollback_snapshot_id_0)

df_time_travel = spark.read \
    .option("snapshot-id", rollback_snapshot_id_0) \
    .table(f"default.{username}_european_cars_time_travel")

df_time_travel.show()

print("Code block completed")

### Rollback Tables Using Snapshots

**What is Rollback in Iceberg?**
Rollback in Iceberg allows you to revert the table's state to a specific snapshot, undoing any subsequent changes. This is useful in scenarios where data corruption, accidental deletion, or unwanted changes occur. By rolling back to a previous snapshot, you can restore the table to its desired state.

**Key Points on Rollback:**
  * Rollback to a Snapshot: You can roll back the table by specifying a snapshotid that corresponds to the point in time you wish to revert to.
  * How it works: The rollback operation rewrites the table to the state of the specified snapshot, effectively "reverting" any changes made after that snapshot.
  * Usage: Rollback can be useful in production environments where you need to ensure data integrity and recover from accidental modifications.

**Code Example:**

In your existing Jupyter notebook add a new cell and run the code below. Examine each statement and it's output to understand how DML operations create snapshots and how we can use these snapshots to do table rollbacks.

In [ ]:
# Drop the table if it exists
spark.sql("DROP TABLE IF EXISTS default.{}_european_cars_rollback".format(username))

# Create an Iceberg table for European cars
spark.sql("""
    CREATE TABLE default.{}_european_cars_rollback (
        car_id STRING,
        car_make STRING,
        car_model STRING,
        car_country STRING
    )
    USING iceberg
""".format(username))

# Insert initial data
spark.sql("""
    INSERT INTO default.{}_european_cars_rollback VALUES 
    ('C001', 'Volkswagen', 'Golf', 'Germany'),
    ('C002', 'BMW', 'X5', 'Germany')
""".format(username))

# Insert new car data (Look, it's not a European Car)
spark.sql("""
    INSERT INTO default.{}_european_cars_rollback VALUES 
    ('C003', 'FORD', 'F150', 'USA')
""".format(username))

print("------------------------")
print("--CURRENT TABLE DATA--")
print("------------------------")

# Show the updated table data
df = spark.sql("SELECT * FROM default.{}_european_cars_rollback".format(username))
df.show(truncate=False)

print("------------------------")
print("--CURRENT SNAPSHOTS REPRESENTING BOTH WRITES--")
print("------------------------")

# List the available snapshots of the table
snapshots_df = spark.sql("""
    SELECT * FROM default.{}_european_cars_rollback.snapshots
""".format(username))
snapshots_df.show()

# Rollback to the initial table state (before the USA Cars)
first_snapshot = snapshots_df.collect()[0].snapshot_id  

print("------------------------")
print("--LETS ROLLBACK TO REMOVE FORD FROM THE TABLE--")
print("------------------------")

# Call the Roll Back Command
spark.sql("CALL spark_catalog.system.rollback_to_snapshot('default.{0}_european_cars_rollback', {1})".format(username,first_snapshot))

# Show the table data after rollback
df = spark.sql("SELECT * FROM default.{}_european_cars_rollback".format(username))
df.show(truncate=False)

print("Code block completed")

## &#x1F600; Well done - You've completed this section of the labs!